# 03 — Evaluation

Dieses Notebook wertet die Baseline-Predictions aus `02_extract.ipynb` gegen mein Hand-Gold aus Phase 2 aus.

Ziel der Phase 3 ist eine erste messbare Baseline: Pro Schema-Feld wird berechnet, wie oft die Modellvorhersage mit meiner manuellen Annotation übereinstimmt. Die schwächsten Felder werden anschließend als Kandidaten für Phase 4 identifiziert.

## Run-Header

| Feld | Wert |
|---|---|
| Datum | 2026-06-01 |
| Predictions-Datei | `../daten/predictions.jsonl` |
| Gold-Datei | `../annotation/meine_gold.csv` |
| Anzeigen-Datei | `../daten/annotations_auswahl.csv` |
| Modell | `Qwen/Qwen2.5-7B-Instruct` |
| Run | Baseline Phase 3 |
| Eval-Entscheidung `skills_top3` | Set-Match, Reihenfolge egal |
| Eval-Entscheidung `gehalt_min_eur` | exakter Match |


## Hypothese vor der Evaluation

Vor der Auswertung erwarte ich, dass `vertragsart`, `gehalt_min_eur` und `gehalt_zeitraum` vergleichsweise stark abschneiden.

`vertragsart` ist in vielen Anzeigen über den Typ der Stelle oder API-nahe Formulierungen relativ klar ableitbar. Bei `gehalt_min_eur` und `gehalt_zeitraum` erwarte ich ebenfalls wenige Fehler, weil in meinem Gold häufig kein konkretes Gehalt genannt ist. Wenn kein Gehalt genannt ist, müssen beide Felder `null` sein.

Schwächer erwarte ich `homeoffice`, `erfahrungslevel` und `skills_top3`.

Bei `homeoffice` können Formulierungen wie „Home Office möglich“, „mobiles Arbeiten“ oder „100 % Home Office“ unterschiedlich interpretiert werden. Bei `erfahrungslevel` ist die Abgrenzung zwischen `junior`, `mid`, `senior`, `egal` und `nicht_genannt` interpretativ, besonders wenn Erfahrung nur indirekt erwähnt wird. Bei `skills_top3` ist zusätzlich schwierig, welche technischen Skills als relevanteste drei ausgewählt werden sollen.

In [1]:
from pathlib import Path
import json
import math
import re

import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

<jemalloc>: Unsupported system page size


## Pfade

In [2]:
# Standard-Pfade im Repository
PRED_PATH = Path("../daten/predictions.jsonl")
GOLD_PATH = Path("../annotation/meine_gold.csv")
ANZEIGEN_PATH = Path("../daten/annotations_auswahl.csv")

print("Predictions:", PRED_PATH)
print("Gold:", GOLD_PATH)
print("Anzeigen:", ANZEIGEN_PATH)

Predictions: ../daten/predictions.jsonl
Gold: ../annotation/meine_gold.csv
Anzeigen: ../daten/annotations_auswahl.csv


## Daten laden

Predictions und Gold werden geladen und die ID-Spalten vereinheitlicht.

In meiner Gold-Datei heißt die Anzeigen-ID teilweise `id`. Für die Evaluation wird sie zu `refnr` umbenannt, weil die Aufgabenstellung den Join über `refnr` vorsieht.

In [3]:
pred = pd.read_json(PRED_PATH, lines=True)
gold = pd.read_csv(GOLD_PATH)
anzeigen = pd.read_csv(ANZEIGEN_PATH)

# ID-Spalte vereinheitlichen
if "id" in gold.columns and "refnr" not in gold.columns:
    gold = gold.rename(columns={"id": "refnr"})

# Whitespace entfernen, weil in CSVs leicht führende/folgende Leerzeichen entstehen
gold["refnr"] = gold["refnr"].astype(str).str.strip()
pred["refnr"] = pred["refnr"].astype(str).str.strip()
anzeigen["refnr"] = anzeigen["refnr"].astype(str).str.strip()

print("Predictions:", pred.shape)
print("Gold:", gold.shape)
print("Anzeigen:", anzeigen.shape)

display(pred.head(3))
display(gold.head(3))

Predictions: (12, 9)
Gold: (12, 8)
Anzeigen: (12, 11)


,refnr,parse_ok,raw_output,homeoffice,vertragsart,erfahrungslevel,gehalt_min_eur,gehalt_zeitraum,skills_top3
0,15939-BB-633097-7878-9999-S,True,"{\n ""homeoffice"": ""nicht_genannt"",\n ""vertragsart"": ""festanstellung"",\n ""erfahrungslevel"": ""mid"",\n ""gehalt_min_...",nicht_genannt,festanstellung,mid,NaN,None,"[Datenanalyse, Datenmodellierung, Auswertung]"
1,15939-BB-633095-7878-7490-S,True,"{\n ""homeoffice"": ""nicht_genannt"",\n ""vertragsart"": ""festanstellung"",\n ""erfahrungslevel"": ""senior"",\n ""gehalt_m...",nicht_genannt,festanstellung,senior,NaN,None,"[OPUS Suite, Data Science, Lifecycle-Analysen]"
2,15939-BB-633455-7878-6343-S,True,"{\n ""homeoffice"": ""nicht_genannt"",\n ""vertragsart"": ""festanstellung"",\n ""erfahrungslevel"": ""mid"",\n ""gehalt_min_...",nicht_genannt,festanstellung,mid,NaN,None,"[Data-Lake, Data-Lineage, Data-Governance]"


,refnr,homeoffice,vertragsart,erfahrungslevel,gehalt_min_eur,gehalt_zeitraum,skills_top3,notiz
0,15939-BB-633097-7878-9999-S,ja,festanstellung,junior,NaN,NaN,FMECA|Reliability Block Diagrams|ILS,"Junior entschieden ohne expliziete Nennung, du..."
1,15939-BB-633095-7878-7490-S,ja,festanstellung,senior,NaN,NaN,OPUS Suite|Python,NaN
2,15939-BB-633455-7878-6343-S,nicht_genannt,festanstellung,senior,NaN,NaN,Data-Lake-/Lakehouse-Architektur|Data-Ingestio...,"festanstellung anhand 30 Tage Urlaub, senior a..."


## Parse-Fails prüfen

Wenn das Modell für eine Anzeige kein valides JSON geliefert hat, zählt diese Anzeige für alle Felder als Fehler. Die Anzahl solcher Fälle wird separat festgehalten, weil Parse-Fails eher auf Prompt- oder Pipeline-Probleme als auf reine Feldinterpretation hindeuten.

In [4]:
if "parse_ok" in pred.columns:
    parse_fail_count = (~pred["parse_ok"].fillna(False)).sum()
else:
    # Wenn keine parse_ok-Spalte vorhanden ist, wird angenommen, dass alle Zeilen parsebar waren.
    pred["parse_ok"] = True
    parse_fail_count = 0

print(f"Parse-Fails: {parse_fail_count} von {len(pred)}")

Parse-Fails: 0 von 12


## Normalisierungsfunktionen

Für die Evaluation müssen Werte vergleichbar gemacht werden.

- Kategoriale Felder werden als String normalisiert.
- Leere Werte, `NaN`, `None`, leere Strings und ähnliche Varianten werden als `None` behandelt.
- `skills_top3` wird als Set verglichen, also ohne Reihenfolge.
- `gehalt_min_eur` wird exakt verglichen. Eine Toleranz verwende ich in der Baseline bewusst nicht, weil das Schema die untere Gehaltsgrenze als konkrete Zahl definiert.

In [5]:
FIELDS = [
    "homeoffice",
    "vertragsart",
    "erfahrungslevel",
    "gehalt_min_eur",
    "gehalt_zeitraum",
    "skills_top3",
]

def is_missing(value):
    """True für NaN/None/leere Strings/null-ähnliche Werte."""
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if isinstance(value, str) and value.strip().lower() in {"", "nan", "none", "null"}:
        return True
    return False

def norm_scalar(value):
    """Normalisiert kategoriale Werte für exakten Vergleich."""
    if is_missing(value):
        return None
    return str(value).strip().lower()

def norm_salary(value):
    """Normalisiert Gehalt für exakten Vergleich."""
    if is_missing(value):
        return None
    try:
        return int(float(value))
    except (ValueError, TypeError):
        return str(value).strip()

def norm_skills(value):
    """
    Normalisiert skills_top3 für Set-Match.
    Gold liegt oft als Pipe-String vor: python|sql|excel.
    Predictions liegen oft als Liste vor: ["Python", "SQL", "Excel"].
    """
    if is_missing(value):
        return set()

    if isinstance(value, list):
        items = value
    elif isinstance(value, str):
        # Falls eine Liste versehentlich als String gespeichert wurde
        stripped = value.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = json.loads(stripped.replace("'", '"'))
                items = parsed if isinstance(parsed, list) else [value]
            except Exception:
                items = value.split("|")
        else:
            items = value.split("|")
    else:
        items = [value]

    return {
        str(item).strip().lower()
        for item in items
        if not is_missing(item)
    }

def field_equal(field, gold_value, pred_value, parse_ok=True):
    """
    Vergleicht ein einzelnes Feld.
    Wenn parse_ok False ist, zählt jedes Feld als falsch.
    """
    if not parse_ok:
        return False

    if field == "skills_top3":
        return norm_skills(gold_value) == norm_skills(pred_value)
    if field == "gehalt_min_eur":
        return norm_salary(gold_value) == norm_salary(pred_value)
    if field == "gehalt_zeitraum":
        return norm_scalar(gold_value) == norm_scalar(pred_value)

    return norm_scalar(gold_value) == norm_scalar(pred_value)


## Join von Gold und Predictions

Predictions und Gold werden über `refnr` verbunden. Dadurch werden nur Anzeigen ausgewertet, die sowohl im Gold als auch in den Predictions vorkommen.

In [6]:
merged = gold.merge(
    pred,
    on="refnr",
    how="left",
    suffixes=("_gold", "_pred"),
    indicator=True,
)

print("Gemergte Zeilen:", len(merged))
print(merged["_merge"].value_counts())

missing_predictions = merged[merged["_merge"] != "both"]["refnr"].tolist()
if missing_predictions:
    print("Für diese Gold-Anzeigen fehlt eine Prediction:")
    print(missing_predictions)

display(merged[["refnr", "_merge"]].head())

Gemergte Zeilen: 12
_merge
both          12
left_only      0
right_only     0
Name: count, dtype: int64


,refnr,_merge
0,15939-BB-633097-7878-9999-S,both
1,15939-BB-633095-7878-7490-S,both
2,15939-BB-633455-7878-6343-S,both
3,15939-BB-633457-7878-2175-S,both
4,18777-931781141-S,both


## Per-Field-Accuracy berechnen

Für jedes Feld wird berechnet:

`Accuracy = n_korrekt / n_total`

Die Tabelle enthält eine Zeile pro Feld mit `Accuracy`, `n_korrekt` und `n_total`.

In [7]:
results = []

for field in FIELDS:
    gold_col = f"{field}_gold"
    pred_col = f"{field}_pred"

    correct_flags = []

    for _, row in merged.iterrows():
        parse_ok = bool(row.get("parse_ok", False)) and row["_merge"] == "both"
        correct = field_equal(
            field,
            row.get(gold_col),
            row.get(pred_col),
            parse_ok=parse_ok,
        )
        correct_flags.append(correct)

    merged[f"{field}_correct"] = correct_flags

    n_correct = int(sum(correct_flags))
    n_total = len(correct_flags)

    results.append({
        "feld": field,
        "accuracy": n_correct / n_total if n_total else 0,
        "n_korrekt": n_correct,
        "n_total": n_total,
    })

eval_table = pd.DataFrame(results).sort_values("accuracy", ascending=True)
display(eval_table)

,feld,accuracy,n_korrekt,n_total
5,skills_top3,0.000000,0,12
2,erfahrungslevel,0.500000,6,12
0,homeoffice,0.583333,7,12
1,vertragsart,0.916667,11,12
3,gehalt_min_eur,0.916667,11,12
4,gehalt_zeitraum,0.916667,11,12


## Zwei schwächste Felder

Die zwei Felder mit der niedrigsten Accuracy sind die wichtigsten Kandidaten für Phase 4. In Phase 4 wird gezielt an einem vermuteten Hebel manipuliert, z. B. Prompt, Schema-Erklärung oder Truncation.

In [8]:
weakest_fields = eval_table.head(2)["feld"].tolist()
print("Zwei schwächste Felder:", weakest_fields)

Zwei schwächste Felder: ['skills_top3', 'erfahrungslevel']


## Fehlerfälle anzeigen

Für die schwächsten Felder werden konkrete Abweichungen angezeigt.

Die Tabelle zeigt pro Fehlerfall:

- `refnr`
- Titel und Firma
- Gold-Wert
- Prediction
- Textausschnitt der Anzeige

In [9]:
# Anzeigeninformationen ergänzen
anzeige_cols = ["refnr"]
for col in ["titel", "firma", "text"]:
    if col in anzeigen.columns:
        anzeige_cols.append(col)

analysis_df = merged.merge(
    anzeigen[anzeige_cols],
    on="refnr",
    how="left",
)

def short_text(text, n=700):
    if is_missing(text):
        return ""
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:n] + ("..." if len(text) > n else "")

for field in weakest_fields:
    print("=" * 100)
    print(f"Fehlerfälle für Feld: {field}")
    print("=" * 100)

    wrong = analysis_df[analysis_df[f"{field}_correct"] == False].copy()

    if wrong.empty:
        print("Keine Fehlerfälle.")
        continue

    rows = []
    for _, row in wrong.iterrows():
        rows.append({
            "refnr": row["refnr"],
            "titel": row.get("titel", ""),
            "firma": row.get("firma", ""),
            "gold": row.get(f"{field}_gold"),
            "prediction": row.get(f"{field}_pred"),
            "parse_ok": row.get("parse_ok", None),
            "textauszug": short_text(row.get("text", ""), n=700),
        })

    display(pd.DataFrame(rows).head(5))


Fehlerfälle für Feld: skills_top3


,refnr,titel,firma,gold,prediction,parse_ok,textauszug
0,15939-BB-633097-7878-9999-S,Data Analyst im Marineschiffbau (m/w/d),Rheinmetall AG,FMECA|Reliability Block Diagrams|ILS,"[Datenanalyse, Datenmodellierung, Auswertung]",True,"Moderne Marineschiffe sind hochkomplexe Systeme, deren Leistungsfähigkeit über Jahrzehnte sichergestellt werden muss..."
1,15939-BB-633095-7878-7490-S,(Senior) Data Analyst - Lifecycle-Analysen im Marineschiffbau (m/w/d),Rheinmetall AG,OPUS Suite|Python,"[OPUS Suite, Data Science, Lifecycle-Analysen]",True,"Marineschiffe sind hochkomplexe Systeme mit Lebenszyklen von 30 Jahren und mehr. Entscheidungen über Wartung, Ersatz..."
2,15939-BB-633455-7878-6343-S,Data Engineer (m/w/d),Rheinmetall AG,Data-Lake-/Lakehouse-Architektur|Data-Ingestio...,"[Data-Lake, Data-Lineage, Data-Governance]",True,Rheinmetall Digital GmbH WHAT WE ARE LOOKING FOR * Konzeption und Implementierung einer strukturierten Lakehouse-Arc...
3,15939-BB-633457-7878-2175-S,MLOps Engineer (m/w/d),Rheinmetall AG,MLOps|Platform Engineering|Model Deployment,"[Machine-Learning, Deployment, Künstliche Intelligenz]",True,Rheinmetall Digital GmbH WOFÜR WIR SIE SUCHEN * Bereitstellung (Deployment) und Betrieb von Machine-Learning-Modelle...
4,18777-931781141-S,Requirements Engineer (m/w/d),OHB-System AG,Siemens Polarion|IBM Rational DOORS|systems en...,"[Siemens Polarion, IBM Rational DOORS, systems engineering]",True,#### Your Tasks - Being the project’s focal point for requirements topics towards customers and suppliers - Ensuring...


Fehlerfälle für Feld: erfahrungslevel


,refnr,titel,firma,gold,prediction,parse_ok,textauszug
0,15939-BB-633097-7878-9999-S,Data Analyst im Marineschiffbau (m/w/d),Rheinmetall AG,junior,mid,True,"Moderne Marineschiffe sind hochkomplexe Systeme, deren Leistungsfähigkeit über Jahrzehnte sichergestellt werden muss..."
1,15939-BB-633455-7878-6343-S,Data Engineer (m/w/d),Rheinmetall AG,senior,mid,True,Rheinmetall Digital GmbH WHAT WE ARE LOOKING FOR * Konzeption und Implementierung einer strukturierten Lakehouse-Arc...
2,15939-BB-633457-7878-2175-S,MLOps Engineer (m/w/d),Rheinmetall AG,senior,mid,True,Rheinmetall Digital GmbH WOFÜR WIR SIE SUCHEN * Bereitstellung (Deployment) und Betrieb von Machine-Learning-Modelle...
3,18777-931781141-S,Requirements Engineer (m/w/d),OHB-System AG,nicht_genannt,junior,True,#### Your Tasks - Being the project’s focal point for requirements topics towards customers and suppliers - Ensuring...
4,13635-7fbe73ac_JB5131141-S,Softwareentwickler:in Machine- / Deep-Learning (Home Office) (m/w/d),zollsoft GmbH,junior,egal,True,"Wo Du mit anpacken kannst • Gleich vom ersten Tag an wird es spannend für Dich, denn Du wirst direkt ein aktiver Tei..."


## Interpretation der schwächsten Felder


### skills_top3

Bei `skills_top3` zeigen sich zwei Fehlertypen. Einerseits gibt es echte Modellfehler, bei denen das Modell eher generische Tätigkeiten wie „Datenanalyse“ oder „Auswertung“ extrahiert, obwohl im Gold spezifischere technische Begriffe wie `FMECA`, `Reliability Block Diagrams` oder `ILS` annotiert wurden. Andererseits gibt es Fälle, in denen Modellvorhersage und Gold inhaltlich sehr nah beieinanderliegen, aber durch unterschiedliche Schreibweisen oder Begriffsvarianten als falsch gewertet werden, z. B. `systems engineering` gegenüber einer leicht abweichenden Gold-Schreibweise.
Es wurde außerdem ein exakter Set-Match verwendet.
Unterschiedliche Schreibweisen oder semantisch ähnliche Begriffe wurden nicht normiert.
Dadurch können einzelne Fehler entstehen, obwohl Gold und Prediction inhaltlich ähnlich sind.
Diese Problematik wird als möglicher Verbesserungshebel für Phase 4 dokumentiert.

Für Phase 4 wäre außerdem ein sinnvoller Hebel, den Prompt für `skills_top3` zu schärfen: Das Modell soll konkrete technische Tools, Methoden, Frameworks und Plattformen bevorzugen und generische Tätigkeitsbeschreibungen vermeiden. Zusätzlich könnte geprüft werden, ob eine Normalisierung der Skill-Schreibweisen sinnvoll ist.

### erfahrungslevel

Bei `erfahrungslevel` liegen die Fehler vor allem an der schwierigen Abgrenzung zwischen `junior`, `mid`, `senior` und `nicht_genannt`. Das Modell tendiert dazu, aus der Komplexität der Aufgaben ein Erfahrungslevel abzuleiten. Dadurch wird z. B. aus anspruchsvollen Data- oder MLOps-Aufgaben häufig `mid`, obwohl im Gold teilweise `senior` annotiert wurde. In anderen Fällen wählt das Modell ein Level, obwohl die Anzeige kein explizites Erfahrungslevel nennt.

Das ist teilweise ein Modellproblem, weil das Modell nicht streng genug zwischen expliziten Anforderungen und impliziten Vermutungen trennt. Gleichzeitig ist es auch ein Schema-Problem, weil die Grenze zwischen `mid`, `senior` und `nicht_genannt` nicht immer eindeutig aus dem Text ableitbar ist.

Für Phase 4 wäre ein sinnvoller Hebel, den Prompt stärker regelbasiert zu formulieren: Das Modell soll `senior` nur bei expliziter Senior-Nennung oder klaren Jahresanforderungen wählen und `nicht_genannt` verwenden, wenn keine konkrete Erfahrungsanforderung im Text steht.


## Erste Hypothese für Phase 4

Aus der Baseline-Evaluation ergeben sich `skills_top3` und `erfahrungslevel` als schwächste Felder.

Meine Hypothese für Phase 4 ist, dass beide Felder durch eine präzisere Prompt-Formulierung verbessert werden können.

Bei `skills_top3` soll der Prompt stärker zwischen konkreten technischen Skills und generischen Tätigkeitsbeschreibungen unterscheiden. Das Modell soll bevorzugt Tools, Methoden, Frameworks, Programmiersprachen und Plattformen extrahieren und Begriffe wie „Datenanalyse“ oder „Auswertung“ nur verwenden, wenn keine spezifischeren technischen Skills genannt werden.

Bei `erfahrungslevel` soll der Prompt verhindern, dass das Modell aus der allgemeinen Komplexität einer Stelle ein Erfahrungslevel ableitet. `senior` soll nur gewählt werden, wenn der Begriff Senior oder eine klare hohe Berufserfahrung genannt wird. `junior` soll nur bei Ausbildung, Berufseinstieg oder explizit geringer Erfahrung gewählt werden. Wenn keine konkrete Aussage zum Erfahrungslevel gemacht wird, soll `nicht_genannt` gewählt werden.

---
# Phase 4 — Iterationen zur Verbesserung der Pipeline

Aus Phase 3 ergeben sich vor allem `skills_top3` und `erfahrungslevel` als schwache Felder.  
In Phase 4 teste ich gezielt zwei Hebel:

1. **Iteration A: Normalisierung von `skills_top3`**  
   Zuerst prüfe ich, ob die schlechte Accuracy bei `skills_top3` teilweise durch Schreibweisen, Groß-/Kleinschreibung oder Formatunterschiede entsteht.

2. **Iteration B: Prompt-Klarstellung**  
   Danach wird ein verschärfter Prompt getestet, der konkretere Regeln für `skills_top3` und `erfahrungslevel` enthält.

Wichtig: Die Baseline-Datei `predictions.jsonl` wird nicht überschrieben. Neue Modellläufe werden unter eigenen Namen gespeichert, z. B. `predictions_iterB.jsonl`.


## Iteration A — Normalisierung + Skill-Level-Auswertung für `skills_top3`

### Hypothese

In der Baseline wird `skills_top3` als **Exact Set-Match** bewertet: Eine Anzeige zählt nur dann als korrekt, wenn alle vorhergesagten Skills exakt mit allen Gold-Skills übereinstimmen. Das ist für ein Top-3-Feld sehr streng. Schon ein einzelner abweichender Skill führt dazu, dass die gesamte Anzeige als falsch zählt.

Deshalb teste ich zuerst den Normierungsweg und ergänze direkt eine fairere Teiltreffer-Metrik:

1. **Normalisierter Exact Set-Match**  
   Schreibweisen, Groß-/Kleinschreibung, Trennzeichen und einfache Alias-Begriffe werden vereinheitlicht. Danach wird weiterhin geprüft, ob das gesamte Skill-Set übereinstimmt.

2. **Skill-Level-Overlap**  
   Zusätzlich wird gezählt, wie viele einzelne Gold-Skills in der Prediction wiedergefunden wurden. Dadurch wird sichtbar, ob das Modell teilweise richtige Skills extrahiert, auch wenn nicht alle drei exakt passen.

3. **Top-3-Slot-Accuracy**  
   Als zusätzliche Sicht wird pro Anzeige mit drei möglichen Skill-Slots gerechnet. Bei 12 Anzeigen ergibt das `n = 36`. Diese Metrik beantwortet: Wie viele der maximal 36 möglichen Top-3-Skill-Treffer wurden erzielt?

Diese Iteration verändert nicht die Modellvorhersagen selbst. Sie prüft, ob die ursprüngliche Bewertung für `skills_top3` zu streng war und ob ein Teil der schwachen Accuracy durch Normierung oder eine passendere Metrik erklärbar ist.

In [12]:
# Iteration A: Normalisierung und Teiltreffer-Metrik für skills_top3


def normalize_skill_iteration_a(skill):
    """
    Normalisiert einzelne Skill-Strings vorsichtig.

    Ziel:
    - offensichtliche Schreibvarianten vereinheitlichen
    - keine freien semantischen Annahmen treffen
    - echte Modellfehler weiterhin sichtbar lassen
    """
    if is_missing(skill):
        return ""

    s = str(skill).lower().strip()

    # Typische Zeichen vereinheitlichen
    s = s.replace("–", "-").replace("—", "-")
    s = s.replace("/", " ")
    s = s.replace("-", " ")
    s = re.sub(r"[\(\)\[\]\{\},.;:]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    aliases = {
        "ms excel": "excel",
        "microsoft excel": "excel",
        "excelkenntnisse": "excel",
        "power point": "powerpoint",
        "microsoft powerpoint": "powerpoint",
        "dev ops": "devops",
        "ci cd": "ci/cd",
        "continuous integration continuous delivery": "ci/cd",
        "system engineering": "systems engineering",
        "systems engineering": "systems engineering",
        "machinelearning": "machine learning",
        "machine learning": "machine learning",
        "artificial intelligence": "ai",
        "künstliche intelligenz": "ai",
        "ki": "ai",
    }

    return aliases.get(s, s)


def norm_skills_iteration_a(value):
    """
    Wandelt skills_top3 in ein normalisiertes Set um.

    Wichtig:
    - Handgold kann Pipe-getrennt sein: Python|SQL|Excel
    - Predictions können als Python-Liste vorliegen: ["Python", "SQL", "Excel"]
    Die Funktion norm_skills aus der Baseline trennt diese Formate bereits.
    Danach werden die Einzelbegriffe zusätzlich normalisiert.
    """
    return {
        normalize_skill_iteration_a(skill)
        for skill in norm_skills(value)
        if normalize_skill_iteration_a(skill)
    }


def field_equal_iteration_a(field, gold_value, pred_value, parse_ok=True):
    """
    Vergleich für Iteration A.

    Nur skills_top3 wird gegenüber der Baseline anders bewertet:
    Es wird ein normalisierter Exact Set-Match verwendet.
    Alle anderen Felder bleiben unverändert.
    """
    if not parse_ok:
        return False

    if field == "skills_top3":
        return norm_skills_iteration_a(gold_value) == norm_skills_iteration_a(pred_value)
    if field == "gehalt_min_eur":
        return norm_salary(gold_value) == norm_salary(pred_value)
    if field == "gehalt_zeitraum":
        return norm_scalar(gold_value) == norm_scalar(pred_value)

    return norm_scalar(gold_value) == norm_scalar(pred_value)


def evaluate_skill_level_overlap(merged_df):
    """
    Zusätzliche Teiltreffer-Auswertung für skills_top3.

    Die klassische Per-Field-Accuracy bleibt erhalten.
    Diese Funktion ergänzt aber eine feinere Metrik:
    - Wie viele einzelne Gold-Skills wurden getroffen?
    - Wie viele der 36 möglichen Top-3-Slots wurden getroffen?
    """
    rows = []

    for _, row in merged_df.iterrows():
        parse_ok = bool(row.get("parse_ok", False)) and row["_merge"] == "both"

        gold_set = norm_skills_iteration_a(row.get("skills_top3_gold")) if parse_ok else set()
        pred_set = norm_skills_iteration_a(row.get("skills_top3_pred")) if parse_ok else set()

        true_positives = len(gold_set & pred_set)
        false_positives = len(pred_set - gold_set)
        false_negatives = len(gold_set - pred_set)

        rows.append({
            "refnr": row["refnr"],
            "gold_original": row.get("skills_top3_gold"),
            "prediction_original": row.get("skills_top3_pred"),
            "gold_normalisiert": sorted(gold_set),
            "prediction_normalisiert": sorted(pred_set),
            "treffer": sorted(gold_set & pred_set),
            "tp": true_positives,
            "fp": false_positives,
            "fn": false_negatives,
            "n_gold_skills": len(gold_set),
            "n_pred_skills": len(pred_set),
            "n_top3_slots": 3,
            "slot_treffer_von_3": true_positives,
        })

    detail_df = pd.DataFrame(rows)

    tp_total = int(detail_df["tp"].sum())
    fp_total = int(detail_df["fp"].sum())
    fn_total = int(detail_df["fn"].sum())
    gold_total = int(detail_df["n_gold_skills"].sum())
    pred_total = int(detail_df["n_pred_skills"].sum())
    slot_total = int(detail_df["n_top3_slots"].sum())

    recall = tp_total / gold_total if gold_total else 0
    precision = tp_total / pred_total if pred_total else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    slot_accuracy = tp_total / slot_total if slot_total else 0

    summary_df = pd.DataFrame([
        {
            "metrik": "Normalisierter Exact Set-Match",
            "wert": float(iterA_table.loc[iterA_table["feld"] == "skills_top3", "accuracy"].iloc[0]),
            "n_korrekt": int(iterA_table.loc[iterA_table["feld"] == "skills_top3", "n_korrekt"].iloc[0]),
            "n_total": int(iterA_table.loc[iterA_table["feld"] == "skills_top3", "n_total"].iloc[0]),
            "interpretation": "Ganze Anzeige zählt nur korrekt, wenn alle Skills als Set übereinstimmen.",
        },
        {
            "metrik": "Skill-Level Recall",
            "wert": recall,
            "n_korrekt": tp_total,
            "n_total": gold_total,
            "interpretation": "Anteil der Gold-Skills, die in der Prediction wiedergefunden wurden.",
        },
        {
            "metrik": "Top-3-Slot-Accuracy",
            "wert": slot_accuracy,
            "n_korrekt": tp_total,
            "n_total": slot_total,
            "interpretation": "Treffer bezogen auf 3 mögliche Skill-Slots pro Anzeige, also n=36 bei 12 Anzeigen.",
        },
        {
            "metrik": "Skill-Level Precision",
            "wert": precision,
            "n_korrekt": tp_total,
            "n_total": pred_total,
            "interpretation": "Anteil der vorhergesagten Skills, die auch im Gold stehen.",
        },
        {
            "metrik": "Skill-Level F1",
            "wert": f1,
            "n_korrekt": tp_total,
            "n_total": gold_total + pred_total,
            "interpretation": "Kombination aus Precision und Recall auf Skill-Ebene.",
        },
    ])

    return detail_df, summary_df


In [14]:
# Hilfsfunktion: Evaluation mit frei wählbarer Vergleichsfunktion

def evaluate_predictions(pred_df, compare_func, label="run"):
    """
    Berechnet dieselbe Per-Field-Accuracy wie in Phase 3,
    aber mit einer frei wählbaren Vergleichsfunktion.

    Damit können Baseline, Normalisierung und spätere Iterationen
    einheitlich verglichen werden.
    """
    merged_run = gold.merge(
        pred_df,
        on="refnr",
        how="left",
        suffixes=("_gold", "_pred"),
        indicator=True,
    )

    run_results = []

    for field in FIELDS:
        gold_col = f"{field}_gold"
        pred_col = f"{field}_pred"

        correct_flags = []

        for _, row in merged_run.iterrows():
            parse_ok = bool(row.get("parse_ok", False)) and row["_merge"] == "both"
            correct = compare_func(
                field,
                row.get(gold_col),
                row.get(pred_col),
                parse_ok=parse_ok,
            )
            correct_flags.append(correct)

        merged_run[f"{field}_correct"] = correct_flags

        n_correct = int(sum(correct_flags))
        n_total = len(correct_flags)

        run_results.append({
            "run": label,
            "feld": field,
            "accuracy": n_correct / n_total if n_total else 0,
            "n_korrekt": n_correct,
            "n_total": n_total,
        })

    return pd.DataFrame(run_results).sort_values("accuracy", ascending=True), merged_run


def compare_eval_tables(base_table, iter_table, iter_name):
    """
    Vergleicht eine Iteration mit der Baseline und berechnet Δ pro Feld.
    """
    base_small = base_table[["feld", "accuracy", "n_korrekt", "n_total"]].rename(
        columns={
            "accuracy": "accuracy_baseline",
            "n_korrekt": "n_korrekt_baseline",
            "n_total": "n_total_baseline",
        }
    )

    iter_small = iter_table[["feld", "accuracy", "n_korrekt", "n_total"]].rename(
        columns={
            "accuracy": f"accuracy_{iter_name}",
            "n_korrekt": f"n_korrekt_{iter_name}",
            "n_total": f"n_total_{iter_name}",
        }
    )

    comparison = base_small.merge(iter_small, on="feld", how="left")
    comparison[f"delta_{iter_name}"] = (
        comparison[f"accuracy_{iter_name}"] - comparison["accuracy_baseline"]
    )

    return comparison.sort_values(f"delta_{iter_name}", ascending=False)


In [15]:
# Iteration A auswerten
# 1. Normalisierter Exact Set-Match für alle Felder
# 2. Zusätzliche Skill-Level-Auswertung für skills_top3

iterA_table, iterA_merged = evaluate_predictions(
    pred,
    field_equal_iteration_a,
    label="Iteration A - Skill-Normalisierung",
)

iterA_delta = compare_eval_tables(eval_table, iterA_table, "iterA")

skill_level_detail, skill_level_summary = evaluate_skill_level_overlap(iterA_merged)

print("Per-Field-Accuracy mit normalisiertem skills_top3-Set-Match:")
display(iterA_table)

print("Δ gegenüber Baseline:")
display(iterA_delta)

print("Zusätzliche Teiltreffer-Auswertung für skills_top3:")
display(skill_level_summary)


Per-Field-Accuracy mit normalisiertem skills_top3-Set-Match:


,run,feld,accuracy,n_korrekt,n_total
5,Iteration A - Skill-Normalisierung,skills_top3,0.000000,0,12
2,Iteration A - Skill-Normalisierung,erfahrungslevel,0.500000,6,12
0,Iteration A - Skill-Normalisierung,homeoffice,0.583333,7,12
1,Iteration A - Skill-Normalisierung,vertragsart,0.916667,11,12
3,Iteration A - Skill-Normalisierung,gehalt_min_eur,0.916667,11,12
4,Iteration A - Skill-Normalisierung,gehalt_zeitraum,0.916667,11,12


Δ gegenüber Baseline:


,feld,accuracy_baseline,n_korrekt_baseline,n_total_baseline,accuracy_iterA,n_korrekt_iterA,n_total_iterA,delta_iterA
0,skills_top3,0.000000,0,12,0.000000,0,12,0.0
1,erfahrungslevel,0.500000,6,12,0.500000,6,12,0.0
2,homeoffice,0.583333,7,12,0.583333,7,12,0.0
3,vertragsart,0.916667,11,12,0.916667,11,12,0.0
4,gehalt_min_eur,0.916667,11,12,0.916667,11,12,0.0
5,gehalt_zeitraum,0.916667,11,12,0.916667,11,12,0.0


Zusätzliche Teiltreffer-Auswertung für skills_top3:


,metrik,wert,n_korrekt,n_total,interpretation
0,Normalisierter Exact Set-Match,0.000000,0,12,"Ganze Anzeige zählt nur korrekt, wenn alle Skills als Set übereinstimmen."
1,Skill-Level Recall,0.205882,7,34,"Anteil der Gold-Skills, die in der Prediction wiedergefunden wurden."
2,Top-3-Slot-Accuracy,0.194444,7,36,"Treffer bezogen auf 3 mögliche Skill-Slots pro Anzeige, also n=36 bei 12 Anzeigen."
3,Skill-Level Precision,0.233333,7,30,"Anteil der vorhergesagten Skills, die auch im Gold stehen."
4,Skill-Level F1,0.218750,7,64,Kombination aus Precision und Recall auf Skill-Ebene.


In [16]:
# Detailansicht: Welche einzelnen Skills wurden getroffen?
# Diese Tabelle ist hilfreich, um Normierungsproblem vs. Modellproblem zu unterscheiden.

skill_level_detail_display = skill_level_detail[[
    "refnr",
    "gold_original",
    "prediction_original",
    "gold_normalisiert",
    "prediction_normalisiert",
    "treffer",
    "slot_treffer_von_3",
    "n_top3_slots",
    "tp",
    "fp",
    "fn",
]].copy()

display(skill_level_detail_display)


,refnr,gold_original,prediction_original,gold_normalisiert,prediction_normalisiert,treffer,slot_treffer_von_3,n_top3_slots,tp,fp,fn
0,15939-BB-633097-7878-9999-S,FMECA|Reliability Block Diagrams|ILS,"[Datenanalyse, Datenmodellierung, Auswertung]","[fmeca, ils, reliability block diagrams]","[auswertung, datenanalyse, datenmodellierung]",[],0,3,0,3,3
1,15939-BB-633095-7878-7490-S,OPUS Suite|Python,"[OPUS Suite, Data Science, Lifecycle-Analysen]","[opus suite, python]","[data science, lifecycle analysen, opus suite]",[opus suite],1,3,1,2,1
2,15939-BB-633455-7878-6343-S,Data-Lake-/Lakehouse-Architektur|Data-Ingestio...,"[Data-Lake, Data-Lineage, Data-Governance]","[data ingestio, data lake lakehouse architektur]","[data governance, data lake, data lineage]",[],0,3,0,3,2
3,15939-BB-633457-7878-2175-S,MLOps|Platform Engineering|Model Deployment,"[Machine-Learning, Deployment, Künstliche Intelligenz]","[mlops, model deployment, platform engineering]","[ai, deployment, machine learning]",[],0,3,0,3,3
4,18777-931781141-S,Siemens Polarion|IBM Rational DOORS|systems en...,"[Siemens Polarion, IBM Rational DOORS, systems engineering]","[ibm rational doors, siemens polarion, systems en]","[ibm rational doors, siemens polarion, systems engineering]","[ibm rational doors, siemens polarion]",2,3,2,1,1
5,13999-k53401.30280-S,Accounting/Controlling|MS Office|ERP-Systeme,[],"[accounting controlling, erp systeme, ms office]",[],[],0,3,0,0,3
6,15939-BB-632493-7878-9058-S,PowerPoint|Excel|Market Reseach,"[Power Point, Excel, Markt-Forschung]","[excel, market reseach, powerpoint]","[excel, markt forschung, powerpoint]","[excel, powerpoint]",2,3,2,1,1
7,13635-7fbe73ac_JB5131141-S,Machine Learning|Python|Linux,"[NLP, Computervision, Spracherkennung]","[linux, machine learning, python]","[computervision, nlp, spracherkennung]",[],0,3,0,3,3
8,20536-lutif851vc-S,Machine Learning|empirical research methods|AI,[],"[ai, empirical research methods, machine learning]",[],[],0,3,0,0,3
9,12826-SA0136034_JB5125696-S,Windows|Linux|Troubleshooting,"[Windows, Linux, MS Office 365]","[linux, troubleshooting, windows]","[linux, ms office 365, windows]","[linux, windows]",2,3,2,1,1


### Befund zu Iteration A

Die Normalisierung prüft zuerst, ob `skills_top3` nur wegen unterschiedlicher Schreibweisen schlecht abschneidet. Der normalisierte Exact Set-Match bleibt weiterhin streng: Eine Anzeige ist nur dann korrekt, wenn alle Skills als Set übereinstimmen.

Die zusätzliche Skill-Level-Auswertung ist feiner. Sie zeigt Teiltreffer, also ob einzelne Skills korrekt erkannt wurden. Dadurch wird sichtbar, ob `0/12` beim Exact Set-Match wirklich bedeutet, dass das Modell gar nichts richtig erkannt hat, oder ob es zwar teilweise richtige Skills findet, aber nicht alle drei exakt passend.

Die **Top-3-Slot-Accuracy** rechnet bewusst mit drei möglichen Skill-Slots pro Anzeige. Bei 12 Anzeigen ergibt das `n = 36`. Diese Metrik passt besser zur Idee eines Top-3-Feldes, weil nicht mehr eine ganze Anzeige falsch wird, sobald nur einer von drei Skills abweicht.

Wenn der normalisierte Exact Set-Match weiterhin niedrig bleibt, aber Skill-Level Recall oder Top-3-Slot-Accuracy höher sind, spricht das dafür, dass die ursprüngliche Metrik zu streng war. Wenn auch die Teiltreffer-Metriken niedrig bleiben, liegt das Problem eher in der inhaltlichen Skill-Auswahl des Modells und sollte in der nächsten Iteration über den Prompt angegangen werden.

### Auswertung Iteration A – Skill-Normalisierung und Teiltreffer-Metrik

Die reine Normalisierung der Skill-Begriffe hat den strengen Exact Set-Match für `skills_top3` nicht verbessert. Auch nach Normalisierung liegt die Accuracy weiterhin bei 0/12. Damit zeigt sich, dass die schlechte Baseline nicht nur durch Groß-/Kleinschreibung, Trennzeichen oder einfache Schreibvarianten erklärbar ist.

Die zusätzliche Skill-Level-Auswertung liefert jedoch ein differenzierteres Bild. Statt ganze Anzeigen nur als vollständig richtig oder falsch zu bewerten, werden hier einzelne Skill-Treffer gezählt. Dabei wurden 7 von 34 Gold-Skills wiedergefunden. Bezogen auf 36 mögliche Top-3-Slots entspricht das einer Slot-Accuracy von ca. 19,4 %. Die Skill-Level-Precision liegt bei ca. 23,3 %, der Recall bei ca. 20,6 %.

Damit zeigt sich: Das Modell trifft einzelne Skills, verfehlt aber häufig die vollständige Top-3-Auswahl. Die Fehler liegen also nicht nur in der Auswertung, sondern vor allem darin, dass das Modell oft andere oder generischere Begriffe auswählt als mein Hand-Gold.

### Diagnose

Iteration A bestätigt teilweise die ursprüngliche Vermutung: Es gibt ein Normierungsproblem, aber es ist nicht die Hauptursache der schlechten `skills_top3`-Accuracy. Die Hauptursache scheint eher ein Modell- bzw. Prompt-Problem zu sein.

Das Modell extrahiert häufig technisch klingende, aber nicht dieselben Begriffe wie im Hand-Gold. Teilweise wählt es generische Tätigkeiten wie `Datenanalyse`, `Datenmodellierung` oder `Auswertung`, obwohl im Gold spezifischere Methoden oder Tools wie `FMECA`, `Reliability Block Diagrams` oder `ILS` stehen.

Für Phase 4 spricht das dafür, als nächste Iteration den Prompt zu schärfen: Das Modell soll konkrete technische Tools, Methoden, Frameworks und Plattformen bevorzugen und generische Tätigkeitsbeschreibungen vermeiden.

## Iteration B — Prompt-Klarstellung für `skills_top3` und `erfahrungslevel`

### Hypothese

Nach Iteration A teste ich eine Prompt-Klarstellung.  
Die Fehleranalyse aus Phase 3 spricht dafür, dass das Modell bei `skills_top3` teilweise zu generische Tätigkeiten extrahiert und bei `erfahrungslevel` zu stark aus der Komplexität der Stelle interpretiert.

Ich erwarte deshalb:

- `skills_top3` verbessert sich, wenn der Prompt konkreter vorgibt, dass technische Tools, Methoden, Frameworks und Plattformen gegenüber generischen Tätigkeiten bevorzugt werden.
- `erfahrungslevel` verbessert sich, wenn der Prompt strenger zwischen expliziten Erfahrungsanforderungen und bloßer Interpretation unterscheidet.

Da die Evaluation nur 12 Anzeigen enthält, entspricht eine einzelne zusätzlich korrekte Anzeige ca. 8,3 Prozentpunkten. Kleine Änderungen müssen deshalb vorsichtig interpretiert werden.


### Prompt-Ergänzung für `02_extract.ipynb`

Für Iteration B wird der System-Prompt in `02_extract.ipynb` um folgende Regeln ergänzt.  
Der neue Run soll anschließend **nicht** als `predictions.jsonl`, sondern z. B. als `predictions_iterB.jsonl` gespeichert werden.

```text
Regeln für erfahrungslevel:
- Wähle "senior" nur, wenn im Titel oder Text explizit "Senior" steht oder eine hohe Berufserfahrung genannt wird, z. B. "mehrjährige Erfahrung", "5 Jahre", "8+ Jahre", "Experte".
- Wähle "mid" nur, wenn konkrete Berufserfahrung verlangt wird, aber keine Senior-Rolle erkennbar ist.
- Wähle "junior" nur bei Ausbildung, Berufseinstieg, Trainee, Praktikum oder ausdrücklich geringer/erster Erfahrung.
- Wähle "egal" nur, wenn ausdrücklich mehrere Level akzeptiert werden, z. B. "Junior bis Senior willkommen".
- Wähle "nicht_genannt", wenn keine konkrete Aussage zum Erfahrungslevel gemacht wird.
- Leite das Erfahrungslevel nicht nur aus der Komplexität der Aufgaben ab.

Regeln für skills_top3:
- Extrahiere maximal 3 konkrete technische Skills, Tools, Methoden, Plattformen, Frameworks oder Programmiersprachen.
- Bevorzuge spezifische Begriffe aus dem Text, z. B. "Python", "OPUS Suite", "FMECA", "IBM Rational DOORS", "MLOps".
- Vermeide generische Tätigkeiten wie "Datenanalyse", "Auswertung", "Recherche", "Reporting", wenn spezifischere technische Skills im Text vorhanden sind.
- Keine Soft Skills, keine Sprachen, keine allgemeinen Studienfächer.
- Gib die Skills als JSON-Liste aus.
```


In [17]:
# Iteration B auswerten, sobald predictions_iterB.jsonl erzeugt wurde

PRED_ITERB_PATH = Path("../daten/predictions_iterB.jsonl")

if PRED_ITERB_PATH.exists():
    pred_iterB = pd.read_json(PRED_ITERB_PATH, lines=True)

    if "refnr" in pred_iterB.columns:
        pred_iterB["refnr"] = pred_iterB["refnr"].astype(str).str.strip()

    if "parse_ok" not in pred_iterB.columns:
        pred_iterB["parse_ok"] = True

    iterB_table, iterB_merged = evaluate_predictions(
        pred_iterB,
        field_equal,
        label="Iteration B - Prompt-Klarstellung",
    )

    display(iterB_table)

    iterB_delta = compare_eval_tables(eval_table, iterB_table, "iterB")
    display(iterB_delta)

else:
    print(f"Noch keine Datei gefunden: {PRED_ITERB_PATH}")
    print("Führe zuerst 02_extract.ipynb mit dem angepassten Prompt aus und speichere die Datei als predictions_iterB.jsonl.")


Noch keine Datei gefunden: ../daten/predictions_iterB.jsonl
Führe zuerst 02_extract.ipynb mit dem angepassten Prompt aus und speichere die Datei als predictions_iterB.jsonl.


In [ ]:
# Optional: Fehlerfälle nach Iteration B anzeigen, wenn die Datei vorhanden ist

if "iterB_merged" in globals():
    iterB_analysis_df = iterB_merged.merge(
        anzeigen[anzeige_cols],
        on="refnr",
        how="left",
    )

    iterB_weakest_fields = iterB_table.head(2)["feld"].tolist()
    print("Schwächste Felder nach Iteration B:", iterB_weakest_fields)

    for field in iterB_weakest_fields:
        print("=" * 100)
        print(f"Fehlerfälle nach Iteration B für Feld: {field}")
        print("=" * 100)

        wrong = iterB_analysis_df[iterB_analysis_df[f"{field}_correct"] == False].copy()

        if wrong.empty:
            print("Keine Fehlerfälle.")
            continue

        rows = []
        for _, row in wrong.iterrows():
            rows.append({
                "refnr": row["refnr"],
                "titel": row.get("titel", ""),
                "firma": row.get("firma", ""),
                "gold": row.get(f"{field}_gold"),
                "prediction": row.get(f"{field}_pred"),
                "parse_ok": row.get("parse_ok", None),
                "textauszug": short_text(row.get("text", ""), n=700),
            })

        display(pd.DataFrame(rows).head(5))
else:
    print("Iteration B wurde noch nicht ausgewertet.")


## Iterations-Tabelle

Die folgende Tabelle fasst Baseline und Iterationen zusammen.

`Δ Gesamt` wird als durchschnittliche Veränderung der Per-Field-Accuracy über alle Felder berechnet.

Für `skills_top3` dokumentiere ich zusätzlich die **Top-3-Slot-Accuracy**, weil der strenge Exact Set-Match bei diesem Feld zu wenig differenziert. Dadurch wird sichtbar, ob einzelne Skills korrekt getroffen wurden, auch wenn die komplette Top-3-Liste nicht exakt übereinstimmt.

In [ ]:
def mean_accuracy(table):
    return float(table["accuracy"].mean())

baseline_mean = mean_accuracy(eval_table)
iterA_mean = mean_accuracy(iterA_table)

baseline_skills_exact = float(eval_table.loc[eval_table["feld"] == "skills_top3", "accuracy"].iloc[0])
iterA_skills_exact = float(iterA_table.loc[iterA_table["feld"] == "skills_top3", "accuracy"].iloc[0])

iterA_slot_accuracy = float(
    skill_level_summary.loc[
        skill_level_summary["metrik"] == "Top-3-Slot-Accuracy",
        "wert",
    ].iloc[0]
)

iterA_skill_recall = float(
    skill_level_summary.loc[
        skill_level_summary["metrik"] == "Skill-Level Recall",
        "wert",
    ].iloc[0]
)

rows = [
    {
        "Iteration": "Baseline",
        "Hypothese": "Ausgangspunkt",
        "Aktion": "Erste Extraktion mit einfachem Prompt und Head-Truncation",
        "Δ Gesamt": "—",
        "Δ schwächstes Feld": "—",
        "Diagnose": "Ausgangspunkt",
    },
    {
        "Iteration": "A: Skill-Normalisierung + Teiltreffer",
        "Hypothese": "Ein Teil der Skill-Fehler entsteht durch Schreibweisen und durch eine zu strenge Exact-Set-Match-Metrik.",
        "Aktion": "Normalisierung von skills_top3; zusätzlich Skill-Level Recall und Top-3-Slot-Accuracy berechnet",
        "Δ Gesamt": round(iterA_mean - baseline_mean, 4),
        "Δ schwächstes Feld": (
            f"Exact Set-Match Δ: {round(iterA_skills_exact - baseline_skills_exact, 4)}; "
            f"Top-3-Slot-Accuracy: {round(iterA_slot_accuracy, 4)}; "
            f"Skill-Level Recall: {round(iterA_skill_recall, 4)}"
        ),
        "Diagnose": "Evaluations-/Postprocessing-Problem, wenn Teiltreffer sichtbar werden; sonst eher Modell-/Prompt-Problem",
    },
]

if "iterB_table" in globals():
    iterB_mean = mean_accuracy(iterB_table)
    iterB_skills = float(iterB_table.loc[iterB_table["feld"] == "skills_top3", "accuracy"].iloc[0])
    rows.append({
        "Iteration": "B: Prompt-Klarstellung",
        "Hypothese": "Strengere Regeln verbessern skills_top3 und erfahrungslevel.",
        "Aktion": "System-Prompt um genauere Regeln ergänzt; neuer Modelllauf als predictions_iterB.jsonl",
        "Δ Gesamt": round(iterB_mean - baseline_mean, 4),
        "Δ schwächstes Feld": round(iterB_skills - baseline_skills_exact, 4),
        "Diagnose": "Modell-/Prompt-Problem, wenn Accuracy steigt; sonst Schema- oder Modellgrenze",
    })
else:
    rows.append({
        "Iteration": "B: Prompt-Klarstellung",
        "Hypothese": "Strengere Regeln verbessern skills_top3 und erfahrungslevel.",
        "Aktion": "System-Prompt um genauere Regeln ergänzen; danach neuer Modelllauf als predictions_iterB.jsonl",
        "Δ Gesamt": "noch offen",
        "Δ schwächstes Feld": "noch offen",
        "Diagnose": "nach Run ergänzen",
    })

iteration_table = pd.DataFrame(rows)
display(iteration_table)


## Synthese

In meiner Baseline gab es keine Parse-Fails. Die größte Schwäche lag daher nicht im JSON-Parsing, sondern in der inhaltlichen Extraktion einzelner Felder.

Bei `skills_top3` zeigte sich zuerst ein Evaluationsproblem: Der Exact Set-Match ist für ein Top-3-Feld sehr streng. Wenn nur ein Skill abweicht, wird die gesamte Anzeige als falsch gezählt. Deshalb habe ich in Iteration A zusätzlich zur Normalisierung eine Skill-Level-Auswertung eingeführt. Diese zählt einzelne Treffer und macht sichtbar, ob das Modell teilweise passende technische Skills extrahiert.

Die Top-3-Slot-Accuracy rechnet mit drei möglichen Skill-Slots pro Anzeige. Bei 12 Anzeigen ergibt das `n = 36`. Diese Sicht ist feiner als `0/12` auf Anzeigenebene und passt besser zur Frage, wie viele der drei gewünschten Skills tatsächlich getroffen wurden.

Wenn trotz Normalisierung und Teiltreffer-Metrik nur wenige Skills getroffen werden, liegt das Problem nicht hauptsächlich an Schreibweisen, sondern an der inhaltlichen Auswahl der Skills durch das Modell. Dann ist der nächste sinnvolle Hebel eine Prompt-Klarstellung oder langfristig ein kontrolliertes Skill-Vokabular.

Bei `erfahrungslevel` liegt die Schwierigkeit vor allem darin, explizite Erfahrungsanforderungen von Modellinterpretationen zu trennen. Das Modell darf nicht allein aus anspruchsvollen Aufgaben ableiten, dass eine Stelle `mid` oder `senior` ist. Hier ist eine Prompt-Klarstellung sinnvoll, aber es bleibt auch ein Schema-Problem: Manche Anzeigen enthalten keine eindeutigen Level-Angaben, obwohl sie sich für Menschen trotzdem nach einem bestimmten Level anfühlen.

Für den Berufsalltag würde ich nicht jede einzelne Skill-Schreibweise manuell perfektionieren. Bei einer Vorstrukturierung von Stellenanzeigen wäre eine robuste Annäherung oft ausreichend. Wenn die extrahierten Daten aber für automatisierte Entscheidungen oder harte Filter genutzt würden, müsste die Schwelle deutlich höher liegen.

Statistisch ist die Auswertung wegen `n=12` vorsichtig zu interpretieren: Eine einzelne Anzeige entspricht bereits ca. 8,3 Prozentpunkten. Die Skill-Level-Auswertung mit bis zu 36 Skill-Slots ist deshalb hilfreicher, weil sie feiner auflöst, sollte aber trotzdem nicht überinterpretiert werden.